# GRUGRU-VAE Hugging Face Demo

This notebook shows how to:
- load the published GRUGRU-VAE model from Hugging Face,
- load example sequences from the LAMP dataset,
- tokenize sequences,
- encode to latent space,
- decode/reconstruct sequences from latent vectors.

In [1]:
from __future__ import annotations

import torch
from datasets import load_dataset
from transformers import AutoModel, AutoTokenizer

torch.set_grad_enabled(False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

/home/pszmk/Latent-Anti-Microbial-Peptides-LAMP/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device(type='cuda')

In [4]:
# ---- Edit these for your run ----
MODEL_ID = "pszmk/lamp-grugru-vae"
MODEL_REVISION = "run-20260427"

TOKENIZER_ID = "pszmk/protein-aa-fast-tokenizer"

DATASET_ID = "pszmk/LAMP-datasets"
DATASET_CONFIG = "nvidia_esm2_uniref_pretraining_data_leq50aa"
DATASET_SPLIT = "validation"
N_SAMPLES = 8
MAX_LENGTH = 64

In [5]:
model = AutoModel.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    trust_remote_code=True,
).to(device)
model.eval()

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID)
dataset = load_dataset(DATASET_ID, DATASET_CONFIG, split=DATASET_SPLIT)

examples = [dataset[i]["sequence"] for i in range(min(N_SAMPLES, len(dataset)))]
examples[:3]

['VSGGRILMRRPGARISLITYWILVH',
 'MSQNVCPRPHPPDECSINEDCGMSMVCCLNDCGRRVCKPAYTPEHIIPRK',
 'RLSMRPTQEELEERNILK']

In [6]:
batch = tokenizer(
    examples,
    padding=True,
    truncation=True,
    max_length=MAX_LENGTH,
    return_tensors="pt",
)
input_ids = batch["input_ids"].to(device)

# Encode sequences into latent distribution parameters.
mean, log_std = model.encode(input_ids)
z = mean  # deterministic latent for demo (no noise sampling)

print("input_ids shape:", tuple(input_ids.shape))
print("mean shape:", tuple(mean.shape))
print("log_std shape:", tuple(log_std.shape))

input_ids shape: (8, 52)
mean shape: (8, 64)
log_std shape: (8, 64)


In [ ]:
# Decode from latent positions (training-style decoder path), then greedy-token reconstruction.
num_steps = input_ids.shape[1] - 1
decoder_out = model.decoder.forward_latent_positions(z=z, num_steps=num_steps)
logits = decoder_out.logits  # [B, T, V]
pred_token_ids = logits.argmax(dim=-1)

reconstructions = [
    tokenizer.decode(ids, skip_special_tokens=True)
    for ids in pred_token_ids.detach().cpu().tolist()
]

for i, (src, rec) in enumerate(zip(examples, reconstructions, strict=False)):
    print(f"[{i}] source: {src}")
    print(f"[{i}] recon : {rec.replace(' ', '')}")
    print("-" * 80)

[0] source: VSGGRILMRRPGARISLITYWILVH
[0] recon : VSGGRILRRRRRAALLLLLLLLLVP
--------------------------------------------------------------------------------
[1] source: MSQNVCPRPHPPDECSINEDCGMSMVCCLNDCGRRVCKPAYTPEHIIPRK
[1] recon : MSQNVCGRRRRRRRCCCCCCCCCCCCCCCCCCCCCCCCSSSSEEEETDRK
--------------------------------------------------------------------------------
[2] source: RLSMRPTQEELEERNILK
[2] recon : RLSKRPTQEEEEEEELLK
--------------------------------------------------------------------------------
[3] source: GYMAPEYAIDGKFSVKSD
[3] recon : GYKAGEFGGGGGGIKKSD
--------------------------------------------------------------------------------
[4] source: MGLGKTIQAIALLSHISETKGIWGPFLV
[4] recon : MGLGKTIQSLAAAALKKKKKGGGGYGLG
--------------------------------------------------------------------------------
[5] source: REALLARGVHGNKANVATARELACWVWAVGLMAEEA
[5] recon : REALLARGARRRRRRRRRRRGGGGGGGGGGGVAVEA
--------------------------------------------------------------------------------
[6] sour

: 